# 01 — Inspecting the CHR timing archives

**What this notebook does.** It opens the simulated CHR (Chronoscope) timing archives: folded pulse
profiles recorded in 500 frequency channels and 126 ten-second sub-integrations per pointing.
It shows how the signal looks in phase, frequency and time. It then demonstrates three physical imprints
that the archives carry: residual dispersion, the 0.1 ms sampling, and interstellar scattering.

**What it needs.** The example dataset folder (the `DATASET` variable below) and the `convening2026` conda
environment, which provides PSRCHIVE and PINT.

**How to run it.** *Kernel → Restart & Run All.* It takes about 2 minutes. Nothing is written to disk
except figures shown inline.

**Everything here is simulated.** An optional `answer_key/` under `data/` holds simulation truth for self-checks;
skip those cells if it is not present.


## Set-up

Sample products are loaded from `data/`. The cell lists archives per pulsar and visit.
If `MANIFEST.csv` is missing, download the CHR sample into `data/` (see `data/README.md`).


In [ ]:
from pathlib import Path
# All sample products live under data/ (download there first; see data/README.md)
from convening2026 import data_dir
from convening2026 import chr_helpers as H
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DATASET = str(data_dir())
DS = Path(DATASET)
man = H.inventory(DS)
print(f"Dataset: {DS}")
print(man.groupby("pulsar").agg(archives=("file", "size"), visits=("visit", "nunique")).to_string())


## 1. What is in the dataset

`README.md` is the data description written at generation time. `MANIFEST.csv` has one row per archive:
its DSA product identifiers (`product_type`, `obs_id`, `dataset_id`), the pulsar, visit, pointing, file
name, start MJD, and the primary-beam response `B_k` of that pointing at the pulsar's position. **What to look for:** P3 has one archive per in-beam pointing in each of its 7 visits.

In [ ]:
readme = (DS / "README.md").read_text()
print(readme.split("## 2.")[0][:3000])        # section 1 of the data description
cols = ["product_type", "obs_id", "pulsar", "visit", "ptg", "discovery", "start_mjd", "B_k", "nbin"]
man[man.pulsar == "P3"][cols].head(12)

## 2. Open one archive

We load P3's visit-0 discovery-pointing archive. **`vap`** (PSRCHIVE's *view archive parameters*) prints
header values; the options are:
- `-c nbin,nchan,...` — the list of parameters to show;
- `nbin` — phase bins;
- `nchan` — frequency channels;
- `nsub` — sub-integrations;
- `dm` — the DM (dispersion measure) the data are dedispersed at;
- `freq`, `bw` — centre frequency and bandwidth (MHz);
- `length` — seconds;
- `name` — the source.

**What to look for:** 128 bins × 500 channels × 126 sub-integrations × 1260 s. The DM is the *search* DM,
which is deliberately very slightly wrong.

In [ ]:
p3 = man[(man.pulsar == "P3") & man.discovery]
p3 = (p3 if len(p3) else man[man.pulsar == "P3"]).iloc[0]
arfile = p3.path
print(H.sh(f"vap -c nbin,nchan,nsub,dm,freq,bw,length,name,mjd {arfile}"))
ar = H.load(arfile)
print("Python API:", ar.get_source(), ar.get_nbin(), "bins,", ar.get_nchan(), "channels,", ar.get_nsubint(), "sub-integrations")


Each archive is a DSA data product of type **DPR-DSA-13** (folded Stokes-I pulsar profile, CHR). Its
identity is the pair (`observationID`, `SRC_NAME`), which is also encoded in the file name:
`dsa13_{observationID}_{SRC_NAME}_v{PRODUCTVER}.psrfits`. The identity keywords sit in the PRIMARY
header:
- `DDSCHEMA` — the version of the DSA data-schema convention;
- `observationID` — the pointing identifier;
- `PRODUCTVER` — the reprocessing version.

The last two are longer than FITS's 8-character limit, so they are stored as `HIERARCH` cards. We read
them with `astropy.io.fits`. **What to look for:** the observationID and version in the header match
the file name.

In [ ]:
from astropy.io import fits
hdr = fits.getheader(arfile, 0)
for key in ("DDSCHEMA", "observationID", "PRODUCTVER", "SRC_NAME", "TELESCOP"):
    print(f"{key:14s} = {hdr[key]!r}")
print("file name     :", Path(arfile).name)

The archive was folded with a single **folding ephemeris**: the pulsar's timing model as known after the
first visit (visit 0). The same file is also stored in `ephemerides/P3/fold.par`. Folding means that every
0.1 ms sample was added into the phase bin predicted by this model. **What to look for:** the spin
frequency `F0`, the binary orbit (`PB` = orbital period in days, `A1` = projected semi-major axis in
light-seconds, `TASC` = epoch of ascending node) and `DM`. The numbers after each value are the fit flag
and the visit-0 uncertainty.

In [ ]:
FOLD = H.fold_ephemeris(DS, "P3")
print(f"fold ephemeris: {FOLD}")
print(FOLD.read_text())


## 3. The pulse profile

Summing all channels (**fscrunch**) and all sub-integrations (**tscrunch**) gives the integrated pulse
profile. **What to look for:** a main peak near phase 0.5 and a weaker interpulse-like component near
0.85. The baseline should be flat, since `remove_baseline` subtracts the off-pulse mean.

In [ ]:
a = H.load(arfile, fscrunch=True, tscrunch=True)
prof = H.data(a)[0, 0]
phase = (np.arange(prof.size) + 0.5) / prof.size
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(phase, prof, drawstyle="steps-mid")
ax.set(xlabel="pulse phase (turns)", ylabel="flux (arbitrary units)",
       title=f"P3, visit 0 discovery pointing: integrated profile (S/N ≈ {H.snr(prof):.0f})")
plt.show()

*Caption:* P3's integrated profile, summed over 800–1107 MHz and 1260 s. The quoted S/N (signal-to-noise
ratio) is a *boxcar* S/N: the best single on-pulse window, divided by the off-pulse rms × √width.

The simulation set this archive's *matched-filter* S/N, the optimum that uses the full profile shape, to
20. That is the discovery S/N assumed by the survey model. A single boxcar misses the weaker component
near phase 0.85, so it reads lower, about 15. PSRCHIVE's `psrstat -c snr` uses yet another estimator and
reads about 11. Always check which S/N definition a number refers to.

## 4. Frequency–phase and time–phase views

These are the Python equivalents of `pav -GTp` (frequency vs phase, time-summed) and `pav -YFp` (time vs
phase, frequency-summed). **What to look for:**
- **Frequency–phase:** the pulse is brighter at low frequency, because of the steep spectrum and the
  wider primary beam there. There is a fine channel-to-channel zig-zag from the integer-sample
  dedispersion.
- **Time–phase:** a vertical stripe that barely moves, because the folding ephemeris is good during
  visit 0.

In [ ]:
aT = H.load(arfile, tscrunch=True)
aF = H.load(arfile, fscrunch=True)
fr = H.freqs(aT)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
axs[0].imshow(H.data(aT)[0], aspect="auto", origin="lower", extent=[0, 1, fr.min(), fr.max()], cmap="afmhot")
axs[0].set(xlabel="pulse phase (turns)", ylabel="frequency (MHz)", title="P3 frequency–phase (tscrunched)")
axs[1].imshow(H.data(aF)[:, 0], aspect="auto", origin="lower", extent=[0, 1, 0, 1260], cmap="afmhot")
axs[1].set(xlabel="pulse phase (turns)", ylabel="time since start (s)", title="P3 time–phase (fscrunched)")
plt.tight_layout(); plt.show()

*Caption:* left, the 500 channels summed over time; right, the 126 sub-integrations summed over
frequency. Both use a linear colour scale.

## 5. S/N versus frequency and versus time

**S/N** here is the boxcar S/N: the sum over the on-pulse window, divided by the off-pulse rms times
√(window width). We find the on-pulse window once, on the integrated profile (a *boxcar search*), and
reuse it for every piece of the data.

A single channel or a single 10 s sub-integration has an S/N of only about 1–2, too noisy to read. We
therefore sum **25 channels** (20 frequency blocks of ≈ 15 MHz) or **9 sub-integrations** (14 time
blocks of 90 s).

**What to look for:**
- **Frequency:** the S/N per block falls with frequency. It follows the model curve
  (ν/953 MHz)^−1.6 × B(ν), where B(ν) is the Gaussian primary beam (FWHM 3.36°/ν_GHz) at this pulsar's
  offset from the pointing centre (`theta_deg` in the manifest); its overall scale is fitted.
- **Time:** the S/N per block is flat within the noise (about ±1). There is no scintillation or RFI in
  this simulation.

In [ ]:
win = H.on_window(prof)                       # on-pulse window of the integrated profile
dT, dF = H.data(aT)[0], H.data(aF)[:, 0]
fb = dT.reshape(20, 25, -1).sum(1); fcen = fr.reshape(20, 25).mean(1)
tb = dF[:126].reshape(14, 9, -1).sum(1); tcen = (np.arange(14) + 0.5) * 90
snr_f = np.array([H.snr(p, win) for p in fb])
snr_t = np.array([H.snr(p, win) for p in tb])
th = p3.theta_deg
model = (fcen / H.NU_REF) ** -1.6 * np.exp(-4 * np.log(2) * th**2 / (3.36 / (fcen / 1e3)) ** 2)
model *= np.sum(snr_f * model) / np.sum(model**2)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
axs[0].errorbar(fcen, snr_f, 1.0, fmt="o", ms=4, label="per 25-channel block (±1)")
axs[0].plot(fcen, model, "r-", label="α = −1.6 × beam (scaled)")
axs[0].set(xlabel="frequency (MHz)", ylabel="S/N per block", title="S/N vs frequency")
axs[0].legend()
axs[1].errorbar(tcen, snr_t, 1.0, fmt="o", ms=4)
axs[1].axhline(snr_t.mean(), color="0.5", lw=0.8)
axs[1].set(xlabel="time since start (s)", ylabel="S/N per 90 s", title="S/N vs time")
plt.tight_layout(); plt.show()
print(f"integrated S/N = {H.snr(prof, win):.1f} (on-pulse window: {win[1]} bins from bin {win[0]})")
print(f"pulsar offset from pointing centre: {th:.2f} deg; beam response at 953 MHz B_k = {p3.B_k:.3f}")
print(f"expected: S/N(block) ~ S/N / sqrt(20) = {H.snr(prof, win) / np.sqrt(20):.1f} (freq), "
      f"S/N / sqrt(14) = {H.snr(prof, win) / np.sqrt(14):.1f} (time)")

*Caption:* left, S/N per 15 MHz block with the expected spectral-index-times-beam shape; right, S/N
per 90 s block, which is constant within its ±1 noise.

## 6. Residual dispersion: recovering the DM

The archive was dedispersed at the *search* DM, which differs from the true DM by about 1 part in 10⁴.
We scan trial DMs: each channel is shifted by the extra dispersion delay relative to 953.35 MHz, the
channels are summed, and we measure how sharp the summed profile becomes. The equivalent command-line
tool is `pdmp` (PSRCHIVE's DM/period optimiser).

**What to look for:**
- The peak of the curve is the best DM.
- The *resolution* is the DM change that moves the pulse by one bin across the band.
- **Precision:** at S/N ≈ 20, one archive measures DM only to about ±0.02 pc cm⁻³ (≈ 4 × 10⁻⁴
  fractional). That is larger than the 1 × 10⁻⁴ error built into the search DM. So we expect the best DM
  to agree with *both* the header and the true DM within the uncertainty. Seeing the 10⁻⁴ offset itself
  would need S/N of several hundred, or many archives combined.

The uncertainty printed below comes from the answer key: the validation step repeated this scan on the
same signal with 8 independent noise draws.

In [ ]:
import json
scan = H.dm_scan(arfile)
AK = H.answer_key_dir(DS, "P3")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(scan["trials"], scan["power"], ".-")
ax.axvline(scan["header"], color="k", ls="--", label=f"header (search) DM {scan['header']:.4f}")
ax.axvline(scan["best"], color="r", label=f"best DM {scan['best']:.4f}")
ax.set(xlabel="trial DM (pc cm⁻³)", ylabel="profile sharpness (normalised)", title="P3 DM scan")
ax.legend(); plt.show()
print(f"best DM = {scan['best']:.4f}; 1-bin resolution {scan['resolution']:.4f}")
print(f"best - header = {scan['best'] - scan['header']:+.4f}")
if AK is None:
    print("answer_key/ not present — skipping truth comparison (optional self-check).")
else:
    val = json.loads((AK / "validation.json").read_text())["dedispersion"]
    sig_dm = val["noisy"]["scatter_std"]
    truth_dm = float(H.read_par(AK / "truth.par")["DM"])
    print(f"best ± σ (answer-key scatter) = {scan['best']:.4f} ± {sig_dm:.4f}")
    print(f"best - true = {scan['best'] - truth_dm:+.4f} ({(scan['best'] - truth_dm) / sig_dm:+.1f} sigma)  [true DM = {truth_dm}]")
    print(f"noise-free archive (answer key): best DM = {val['noise_free']['dm_best']:.5f}")


*Caption:* profile sharpness vs trial DM for P3. The red line and band are the recovered DM and its
1σ uncertainty; the dashed line is the DM used for dedispersion.

## 7. Sampling, smearing and scattering imprints

The archives were folded from 0.1 ms samples of 122.68 kHz channels. Every channel profile is therefore
the intrinsic pulse convolved with three kernels:
- **the 0.1 ms sampling boxcar** — 1.6 % of P3's 6.1 ms period, about 2 of its 128 bins (grey bar in the
  plot). No feature can be narrower than that.
- **intra-channel dispersion smearing** — the DM delay changes across one native channel. At DM 60 this is
  0.12 ms at 800 MHz but 0.05 ms at 1107 MHz (it scales as ν⁻³).
- **scattering** — the Bhat et al. (2004) relation gives only ≈ 2 μs at DM 60. It matters for the
  high-DM pulsar types, which are not in this preview dataset.

The cell below prints each kernel's width at the band edges, next to the pulse's own width
(W_eq ≈ 0.61 ms = 10 % of P). **What to look for:** all three kernels are small compared with the pulse.
The bottom and top of the band therefore look the same within the noise, and at S/N 20 the ≈ 1 %
broadening at 800 MHz cannot be measured. The kernels still matter for timing precision at high S/N, and
for the fast and high-DM pulsar types.

In [ ]:
a = H.load(arfile, tscrunch=True)
d = H.data(a)[0]; f = H.freqs(a)
lo, hi = d[f < 875].sum(0), d[f > 1030].sum(0)
P3 = a.get_Integration(0).get_folding_period()
ph = (np.arange(lo.size) + 0.5) / lo.size
w = 1e-4 / P3
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(ph, lo / lo[win[0]:win[0] + win[1]].sum(), drawstyle="steps-mid", label="800–875 MHz")
ax.plot(ph, hi / hi[win[0]:win[0] + win[1]].sum(), drawstyle="steps-mid", label="1030–1107 MHz")
ax.axvspan(0.40, 0.40 + w, color="0.6", label=f"0.1 ms sample = {w:.3f} turn")
ax.set(xlim=(0.35, 0.65), xlabel="pulse phase (turns)", ylabel="flux / on-pulse sum",
       title="P3 main component: bottom vs top of the band (area-normalised)")
ax.legend(fontsize=8); plt.show()
dm = float(H.read_par(H.fold_ephemeris(DS, "P3"))["DM"])
for nu in (800.0, 1106.7):
    smear = 8.3e-3 * dm * 0.12268 / (nu / 1e3) ** 3                      # ms
    x = np.log10(dm)
    scat = 10 ** (-6.46 + 0.154 * x + 1.07 * x**2 - 3.86 * np.log10(nu / 1e3))   # ms
    print(f"{nu:6.1f} MHz: sampling 0.100 ms, DM smearing {smear:.3f} ms, scattering {scat * 1e3:.1f} us "
          f"vs pulse W_eq {0.10 * P3 * 1e3:.2f} ms")


*Caption:* P3's main pulse component from the lowest and highest 75 MHz of the band, each normalised
to unit on-pulse area; the grey bar is one 0.1 ms sample.

## 8. Data volume

`volume.json` records the size of this dataset and a projection for every pulsar type over all 7 visits.
It was produced by `chrpreview estimate`. **What to look for:** sizes scale with the number of phase
bins (128/256/512). The 4-byte column corresponds to the real CHR product format.

In [ ]:
import json
vol = json.loads((DS / "volume.json").read_text())
print(f"this dataset: {vol['generated']['n_archives']} archives, {vol['generated']['total_bytes'] / 1e9:.2f} GB")
t = pd.DataFrame(vol["projection_16bit"]["per_pulsar"])[["pulsar", "P_ms", "nbin", "n_archives", "bytes_total"]]
t["GB (16-bit)"] = t.pop("bytes_total") / 1e9
t["GB (4-byte)"] = [r["bytes_total"] / 1e9 for r in vol["projection_32bit"]["per_pulsar"]]
print(t.round(3).to_string(index=False))
print(f"total: {t['GB (16-bit)'].sum():.2f} GB (16-bit), {t['GB (4-byte)'].sum():.2f} GB (4-byte)")

## What can go wrong

- **`FileNotFoundError: No MANIFEST.csv`** — `data/` has no `MANIFEST.csv`. Download the CHR sample into `data/` — see
  the Set-up cell and re-run from the top.
- **`ModuleNotFoundError: psrchive`** — the notebook is not using the `convening2026` kernel. Choose it under
  *Kernel → Change kernel*.
- **`vap -c site` prints `INVALID`** — harmless. The site is a GBT stand-in, and conda PSRCHIVE has no TEMPO
  observatory table; PINT and `pat` still identify it correctly.
- **An all-black or all-white image** — you plotted before `remove_baseline`, so one bright channel
  dominates. Use `H.load`, which removes the baseline.


## Try next

- Repeat Sections 3–5 for a late-visit P3 archive (`visit == 6`) and watch the pulse drift across phase: the visit-0 ephemeris is no longer exact there.
- Plot the pulse phase of every P3 archive against orbital phase (from `PB` and `TASC` in `fold.par`) to see the Rømer delay the folding ephemeris removes.
- Compare the S/N of the six visit-0 archives with their beam responses `B_k` in `MANIFEST.csv`.
- Continue with `dsa13_timing.ipynb` to turn these archives into times of arrival and a timing solution.